# OpenMM/OpenCL 50k Sparse MD Preview

This lightweight notebook previews the 50,000-step OpenMM CHARMM/PME run sampled into 11 frames. It uses the existing processed artifact and generated HTML viewer instead of rebuilding the visualization in notebook memory.

In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import HTML, Markdown, display

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "helpers").exists():
    NOTEBOOK_DIR = Path("notebooks/ligand-receptor-motion").resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from helpers.motion_analysis import (  # noqa: E402
    ProcessedTrajectory,
    analysis_tables,
    contact_counts,
    ligand_com_displacement,
    trajectory_quality_report,
    water_counts_around_ligand,
)

DATASET_NAME = os.environ.get("OPENMM_MD_DATASET", "729-50000-opencl-charmm-pme-sample11")
DATA_DIR = NOTEBOOK_DIR / "data/openmm-md" / DATASET_NAME
TRAJECTORY_PATH = DATA_DIR / "processed_trajectory.npz"
REPORT_PATH = DATA_DIR / "openmm_charmm_md_run_report.json"
SUMMARY_PATH = DATA_DIR / "preview_summary.json"
HTML_PREVIEW_PATH = DATA_DIR / "ligand_motion_preview.html"

## Artifact Summary

In [ ]:
missing = [path for path in (TRAJECTORY_PATH, REPORT_PATH, SUMMARY_PATH, HTML_PREVIEW_PATH) if not path.exists()]
if missing:
    raise FileNotFoundError("Missing OpenMM sparse preview artifacts: " + ", ".join(map(str, missing)))

summary = json.loads(SUMMARY_PATH.read_text())
report = json.loads(REPORT_PATH.read_text())
display(
    pd.DataFrame(
        [
            {
                "dataset": DATASET_NAME,
                "engine": report.get("engine"),
                "workflow": report.get("workflow"),
                "platform": summary.get("platform"),
                "device": summary.get("platform_properties", {}).get("DeviceName"),
                "precision": summary.get("platform_properties", {}).get("Precision"),
                "atoms": summary.get("atoms"),
                "steps": summary.get("steps"),
                "frames": summary.get("frames"),
                "sample_interval": summary.get("sample_interval"),
                "simulated_time_ps": summary.get("simulated_time_ps"),
                "wall_seconds": summary.get("elapsed_wall_seconds"),
                "steps_per_second": summary.get("steps_per_second"),
                "final_ligand_displacement_A": summary.get("final_ligand_displacement_A"),
                "max_ligand_displacement_A": summary.get("max_ligand_displacement_A"),
            }
        ]
    )
)

display(Markdown(f"Preview HTML: `{HTML_PREVIEW_PATH}`"))

## Lightweight Motion Metrics

In [ ]:
trajectory = ProcessedTrajectory.load(TRAJECTORY_PATH)
displacement = ligand_com_displacement(trajectory)
contacts = contact_counts(trajectory)
waters = water_counts_around_ligand(trajectory)

frame_table = pd.DataFrame(
    {
        "frame": range(trajectory.frame_count),
        "time_ps": trajectory.time_ps,
        "ligand_com_displacement_A": displacement,
        "contact_count_4A": contacts,
        "water_count_5A": waters,
    }
)
display(frame_table)

display(
    px.line(
        frame_table,
        x="time_ps",
        y=["ligand_com_displacement_A", "water_count_5A"],
        markers=True,
        title="Sparse-frame ligand motion and nearby water count",
    )
)
display(
    px.line(
        frame_table,
        x="time_ps",
        y="contact_count_4A",
        markers=True,
        title="Ligand-receptor contact count across sampled frames",
    )
)

quality = trajectory_quality_report(trajectory, max_constraint_error_A=0.0)
display(pd.DataFrame([{k: v for k, v in quality.items() if k != "warnings"}]))
for warning in quality["warnings"]:
    display(Markdown(f"- {warning}"))

## Embedded 3D Preview

The embedded viewer inlines the pre-generated HTML file. This avoids brittle JupyterLab iframe path resolution while still avoiding Plotly scene construction inside this notebook kernel.

In [ ]:
display(HTML(filename=str(HTML_PREVIEW_PATH)))

## Regenerate Artifact

```bash
uv run python scripts/run_openmm_gpcrmd_charmm_md.py \
  --cache-dir notebooks/ligand-receptor-motion/data/gpcrmd-cache/729 \
  --prepared-dir notebooks/ligand-receptor-motion/data/gpcrmd-mlx/729-50steps-sample50 \
  --out notebooks/ligand-receptor-motion/data/openmm-md/729-50000-opencl-charmm-pme-sample11 \
  --platform OpenCL \
  --steps 50000 --sample-interval 5000 \
  --dt-ps 0.0001 --minimize-steps 2000 \
  --temperature 310 --friction 0.1 --positions-source prepared
```